# 2026-10-21_12 summary：两对象 celltype/age UMAP + marker dotplot

**需求**（参考 `2026-05-25_create_summary/2026-07-28_成年summary.ipynb` 风格）：
- 两个对象 = development（GSE142526 D125C/D125P，F6 官方注释）+ adult（GSE277326+GSE155288 整合重注释对象，2026-10-21_8）
- 5 张图：development celltype UMAP、adult celltype UMAP、adult age UMAP、development marker dotplot、adult marker dotplot
- development celltype UMAP **不含 T1/T3、T2、Midbrain**（2026-10-25 修改：仅显示 PR/RPC/AC/BC/HC/MG/RGC/Micro，UMAP 坐标不变）
- celltype 颜色使用用户指定 `cols.celltype`（Paired 系）；胚胎特有类型补色：PR=#6A3D9A（未拆分光感受器）、T1/T3=#BDBDBD、T2=#969696、Midbrain=#252525；Prog/Prog/Glia 合并为 RPC，miG→Micro
- 年龄颜色沿用本项目 5 龄渐变；仅 PDF

**输出**：
- `Data/2026-10-21_12_development_object.rds`（胚胎处理对象，harmony+UMAP）
- `Plot/2026-10-21_12_{development,adult}_umap_celltype.pdf`、`Plot/2026-10-21_12_adult_umap_age.pdf`
- `Plot/2026-10-21_12_{development,adult}_marker.pdf`

In [ ]:
# ---- 1. 环境 ---------------------------------------------------------------
suppressPackageStartupMessages({
  library(Seurat)
  library(harmony)
  library(Matrix)
  library(dplyr)
  library(ggplot2)
})
set.seed(1)

proj_dir <- "/project/work/2026-10-20_other_dataset"
data_dir <- file.path(proj_dir, "Data")
plot_dir <- file.path(proj_dir, "Plot")
date_str <- "2026-10-21_12"

In [ ]:
# ---- 2. 颜色（celltype 用用户指定配色; 年龄用本项目渐变）--------------------
cols.celltype <- c(
  Rod   = "#A6CEE3",
  Cone  = "#1F78B4",
  AC    = "#B2DF8A",
  BC    = "#33A02C",
  HC    = "#FB9A99",
  MG    = "#E31A1C",
  RGC   = "#FDBF6F",
  RPC   = "#FF7F00",
  Micro = "#66C2A5",
  Astro = "#FC8D62"
)
## 胚胎特有类型补色（不在用户配色内的：PR=未拆分光感受器, T1/T3/T2=过渡态, Midbrain=污染）
cols.dev.extra <- c(PR = "#6A3D9A", "T1/T3" = "#BDBDBD", T2 = "#969696", Midbrain = "#252525")
cols.age <- c("21y" = "#3288BD", "27y" = "#66C2A5", "43y" = "#A6D96A",
              "78y" = "#FDAE61", "90y" = "#D73027")
cat("配色就绪\n")

In [ ]:
# ---- 3. 构建 development 对象（GSE142526 D125C/D125P + F6 官方注释）--------
dev_dir <- file.path(proj_dir, "00data/2026-10-20_development_new/GSE142526")
dev_meta <- read.csv(file.path(data_dir, "2026-10-20_3_GSE142526_metadata.csv"),
                     stringsAsFactors = FALSE)

so_list <- list()
for (smp in c("D125C", "D125P")) {
  genes <- read.delim(file.path(dev_dir, smp, "GRCh38", "genes.tsv"), header = FALSE,
                      col.names = c("ensembl", "symbol"), stringsAsFactors = FALSE)
  m <- readMM(file.path(dev_dir, smp, "GRCh38", "matrix.mtx"))
  bc <- readLines(file.path(dev_dir, smp, "GRCh38", "barcodes.tsv"))
  m <- as(m, "CsparseMatrix")
  rownames(m) <- make.unique(genes$symbol)
  colnames(m) <- paste0(smp, "_", sub("-1$", "", bc))
  so_list[[smp]] <- CreateSeuratObject(counts = m)
  rm(m); invisible(gc())
}
obj_dev <- merge(so_list[[1]], so_list[[2]])
obj_dev <- JoinLayers(obj_dev)
rm(so_list); invisible(gc())

mm <- dev_meta[match(colnames(obj_dev), dev_meta$cell_id),
               c("sample_id", "region", "tissue", "celltype_major")]
obj_dev <- AddMetaData(obj_dev, mm)
stopifnot(!any(is.na(obj_dev$celltype_major)))
obj_dev$donor_age <- "FD125"
obj_dev$percent.mt <- PercentageFeatureSet(obj_dev, pattern = "^MT-")

## 展示用 celltype（对齐用户配色：Prog/Prog/Glia -> RPC, miG -> Micro）
disp_map <- c(PR = "PR", BC = "BC", AC = "AC", HC = "HC", MG = "MG", RGC = "RGC",
              Prog = "RPC", "Prog/Glia" = "RPC", miG = "Micro",
              "T1/T3" = "T1/T3", T2 = "T2", Midbrain = "Midbrain")
obj_dev$celltype <- factor(unname(disp_map[as.character(obj_dev$celltype_major)]),
                           levels = c("PR","RPC","AC","BC","HC","MG","RGC","Micro",
                                      "T1/T3","T2","Midbrain"))
cat("development 对象:", ncol(obj_dev), "cells x", nrow(obj_dev), "genes\n")
print(table(obj_dev$celltype))

In [ ]:
# ---- 4. development 处理（Norm -> HVG -> Scale -> PCA -> harmony -> UMAP）----
options(future.globals.maxSize = 16 * 1024^3)
future::plan("multisession", workers = 8)
obj_dev <- NormalizeData(obj_dev, verbose = FALSE)
obj_dev <- FindVariableFeatures(obj_dev, nfeatures = 3000, verbose = FALSE)
obj_dev <- ScaleData(obj_dev, vars.to.regress = "percent.mt", verbose = FALSE)
obj_dev <- RunPCA(obj_dev, npcs = 30, verbose = FALSE)
future::plan("sequential")
obj_dev <- RunHarmony(obj_dev, group.by.vars = "sample_id", reduction.use = "pca",
                      dims.use = 1:30, plot.convergence = FALSE)
obj_dev <- RunUMAP(obj_dev, reduction = "harmony", dims = 1:20, verbose = FALSE)
saveRDS(obj_dev, file.path(data_dir, paste0(date_str, "_development_object.rds")))
cat("development 对象已保存\n")

In [ ]:
# ---- 5. 图1: development celltype UMAP（去掉 T1/T3、T2、Midbrain）----------
remove_types <- c("T1/T3", "T2", "Midbrain")
keep_cells <- colnames(obj_dev)[!as.character(obj_dev$celltype) %in% remove_types]
obj_dev_plot <- subset(obj_dev, cells = keep_cells)
obj_dev_plot$celltype <- droplevels(obj_dev_plot$celltype)
cat("去除", paste(remove_types, collapse = "/"), "| 保留细胞:",
    length(keep_cells), "/", ncol(obj_dev), "\n")
print(table(obj_dev_plot$celltype))

cols_dev_all <- c(cols.celltype, cols.dev.extra)[levels(obj_dev_plot$celltype)]
p1 <- DimPlot(obj_dev_plot, reduction = "umap", group.by = "celltype",
              label = TRUE, cols = cols_dev_all, raster = TRUE)
pdf(file.path(plot_dir, paste0(date_str, "_development_umap_celltype.pdf")),
    width = 8, height = 8, useDingbats = FALSE)
print(p1)
dev.off()
cat("图1 done\n")

In [ ]:
# ---- 6. 图2: adult celltype UMAP（对象 = 2026-10-21_8 重注释整合对象）-------
adu <- readRDS(file.path(data_dir, "2026-10-21_8_adult_integrated_retina7_reannotated.rds"))
adu$celltype_plot <- factor(as.character(adu$celltype_reanno),
                            levels = c("Rod", "Cone", "AC", "BC", "HC", "MG", "RGC"))
p2 <- DimPlot(adu, reduction = "umap", group.by = "celltype_plot",
              label = TRUE, cols = cols.celltype, raster = TRUE)
pdf(file.path(plot_dir, paste0(date_str, "_adult_umap_celltype.pdf")),
    width = 8, height = 8, useDingbats = FALSE)
print(p2)
dev.off()
cat("图2 done | adult:", ncol(adu), "cells\n")

In [ ]:
# ---- 7. 图3: adult age UMAP（21y-90y；对象现状包含21y）---------------------
p3 <- DimPlot(adu, reduction = "umap", group.by = "donor_age",
              label = FALSE, cols = cols.age, raster = TRUE)
pdf(file.path(plot_dir, paste0(date_str, "_adult_umap_age.pdf")),
    width = 8, height = 8, useDingbats = FALSE)
print(p3)
dev.off()
cat("图3 done\n")

In [ ]:
# ---- 8. marker dotplot 函数（参考 成年summary cell 51-62 样式）-------------
make_marker_dotplot <- function(obj, group.by, markers, types, pdf_file, w = 12, h = 5) {
  keep <- markers %in% rownames(obj)
  if (any(!keep)) {
    cat("  剔除对象中不存在的marker:", paste(markers[!keep], collapse = ","), "\n")
    markers <- markers[keep]; types <- types[keep]
  }
  df_map <- data.frame(features.plot = markers, type = types)
  p <- DotPlot(obj, features = markers, group.by = group.by)
  use.data <- merge(p$data, df_map, by = "features.plot")
  use.data$features.plot <- factor(use.data$features.plot, levels = markers)
  use.data$type <- factor(use.data$type, levels = unique(types))
  p1 <- ggplot(use.data, aes(x = features.plot, y = id)) +
    geom_point(aes(size = pct.exp, color = avg.exp.scaled)) +
    scale_size(range = c(0.1, 5)) +
    scale_color_gradientn(colours = c("#313695", "#762A83", "#D6604D", "#FFFF00"),
                          name = "Scaled\nExpression") +
    facet_wrap(~ type, scales = "free_x", nrow = 1) +
    theme_classic() +
    theme(text = element_text(color = "black"),
          axis.text.y = element_text(color = "black", size = 12),
          axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1,
                                     color = "black", size = 10),
          legend.title = element_text(size = 10),
          strip.text = element_text(size = 11)) +
    labs(x = "Features", y = "Cell Type",
         size = "Percentag\n Expressed", color = "Scaled\nExpression")
  pdf(pdf_file, width = w, height = h, useDingbats = FALSE)
  print(p1)
  dev.off()
  cat("done:", basename(pdf_file), "\n")
}
cat("dotplot 函数就绪\n")

In [ ]:
# ---- 9. 图4: development marker dotplot（8 个有注释marker的类型）----------
dot_types <- c("PR","RPC","AC","BC","HC","MG","RGC","Micro")
obj_dev_dot <- subset(obj_dev,
  cells = colnames(obj_dev)[obj_dev$celltype %in% dot_types])
obj_dev_dot$celltype_dot <- droplevels(factor(as.character(obj_dev_dot$celltype),
                                              levels = dot_types))
dev_markers <- c("RCVRN","CRX","ARR3","RHO",              # PR
                 "SOX2","SLC1A3","HES1",                   # RPC
                 "GAD1","GAD2","TFAP2B",                   # AC
                 "VSX2","ISL1","CABP5",                    # BC
                 "LHX1","PVALB","CALB1",                   # HC
                 "RLBP1","GLUL","SOX9",                    # MG
                 "RBPMS","POU4F2","NEFM",                  # RGC
                 "PTPRC","AIF1","C1QB")                    # Micro
dev_types <- c(rep("PR",4), rep("RPC",3), rep("AC",3), rep("BC",3),
               rep("HC",3), rep("MG",3), rep("RGC",3), rep("Micro",3))
make_marker_dotplot(obj_dev_dot, "celltype_dot", dev_markers, dev_types,
  file.path(plot_dir, paste0(date_str, "_development_marker.pdf")))
cat("图4 done\n")

In [ ]:
# ---- 10. 图5: adult marker dotplot（沿用 成年summary marker 清单）-----------
adu_markers <- c("RHO","PDE6A","CNGA1","SAG",             # Rod
                 "OPN1MW","OPN1LW","OPN1SW","GNAT2",       # Cone
                 "GAD1","GAD2","SLC6A9",                   # AC
                 "VSX1","VSX2","GRM6","TRPM1",             # BC
                 "ONECUT1","ONECUT2","LHX1","CALB1",       # HC
                 "RLBP1","RGR","DKK3",                     # MG
                 "RBPMS","POU4F2","SLC17A6","NEFM")        # RGC
adu_types <- c(rep("Rod",4), rep("Cone",4), rep("AC",3), rep("BC",4),
               rep("HC",4), rep("MG",3), rep("RGC",4))
make_marker_dotplot(adu, "celltype_plot", adu_markers, adu_types,
  file.path(plot_dir, paste0(date_str, "_adult_marker.pdf")))
cat("图5 done\n")

In [ ]:
# ---- 11. 文件清单与 sessionInfo ---------------------------------------------
out_files <- c(
  file.path(data_dir, paste0(date_str, "_development_object.rds")),
  file.path(plot_dir, paste0(date_str, "_development_umap_celltype.pdf")),
  file.path(plot_dir, paste0(date_str, "_adult_umap_celltype.pdf")),
  file.path(plot_dir, paste0(date_str, "_adult_umap_age.pdf")),
  file.path(plot_dir, paste0(date_str, "_development_marker.pdf")),
  file.path(plot_dir, paste0(date_str, "_adult_marker.pdf")))
fi <- do.call(rbind, lapply(out_files, function(f)
  data.frame(file = basename(f), size_MB = round(file.size(f) / 1024^2, 3))))
print(fi, row.names = FALSE)
cat("\n全部完成。\n\n")
sessionInfo()

## 追加（2026-10-25）：adult 三图去 21y 版本（`_no21y` 后缀，原图保留）

- **去除 21y 供体**（Macula_21y/Retina_21y，质量异常见 README 数据质控记录），adult 对象仅显示/计算
  27y/43y/78y/90y 细胞（UMAP 坐标不变，仅不显示 21y 细胞）
- **age UMAP 换用 `00_colors.R` 中未使用的颜色**：`#3288BD`(27y) → `#A6D96A`(43y) → `#FDAE61`(78y) →
  `#D73027`(90y)（逐 hex 核对不在 00_colors.R 已用色中；保持年轻→老渐变）
- 三张图同名加 `_no21y` 后缀输出，原 `2026-10-21_12_adult_*.pdf` 不改动

In [ ]:
# ---- 12. adult 三图去 21y 版本 ----
adu <- readRDS(file.path(data_dir, "2026-10-21_8_adult_integrated_retina7_reannotated.rds"))
keep_no21 <- colnames(adu)[as.character(adu$donor_age) != "21y"]
adu_no21 <- subset(adu, cells = keep_no21)
adu_no21$donor_age <- droplevels(factor(as.character(adu_no21$donor_age),
                                       levels = c("27y", "43y", "78y", "90y")))
adu_no21$celltype_plot <- factor(as.character(adu_no21$celltype_reanno),
                                 levels = c("Rod", "Cone", "AC", "BC", "HC", "MG", "RGC"))
cat("去21y后细胞数:", ncol(adu_no21), "/", ncol(adu), "\n")
print(table(adu_no21$donor_age))

## age 颜色：00_colors.R 中未使用的颜色（逐hex核对）
cols.age.no21 <- c("27y" = "#3288BD", "43y" = "#A6D96A",
                   "78y" = "#FDAE61", "90y" = "#D73027")

## 图2'：celltype UMAP（无21y，坐标不变）
p2n <- DimPlot(adu_no21, reduction = "umap", group.by = "celltype_plot",
               label = TRUE, cols = cols.celltype, raster = TRUE)
pdf(file.path(plot_dir, paste0(date_str, "_adult_umap_celltype_no21y.pdf")),
    width = 8, height = 8, useDingbats = FALSE)
print(p2n)
dev.off()

## 图3'：age UMAP（无21y，新配色）
p3n <- DimPlot(adu_no21, reduction = "umap", group.by = "donor_age",
               label = FALSE, cols = cols.age.no21, raster = TRUE)
pdf(file.path(plot_dir, paste0(date_str, "_adult_umap_age_no21y.pdf")),
    width = 8, height = 8, useDingbats = FALSE)
print(p3n)
dev.off()
cat("UMAP 两图 done\n")

In [ ]:
# ---- 13. 图5'：adult marker dotplot（无21y细胞参与计算）----
adu_markers <- c("RHO","PDE6A","CNGA1","SAG",
                 "OPN1MW","OPN1LW","OPN1SW","GNAT2",
                 "GAD1","GAD2","SLC6A9",
                 "VSX1","VSX2","GRM6","TRPM1",
                 "ONECUT1","ONECUT2","LHX1","CALB1",
                 "RLBP1","RGR","DKK3",
                 "RBPMS","POU4F2","SLC17A6","NEFM")
adu_types <- c(rep("Rod",4), rep("Cone",4), rep("AC",3), rep("BC",4),
               rep("HC",4), rep("MG",3), rep("RGC",4))
make_marker_dotplot(adu_no21, "celltype_plot", adu_markers, adu_types,
  file.path(plot_dir, paste0(date_str, "_adult_marker_no21y.pdf")))
cat("图5' done\n")